In [ ]:
# ==============================================================================
# KROK 1: Generujemy nieliniowe zjawisko (np. cykl koniunkturalny / krzywa popytu)
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Ustawienia ziarna i estetyki wykresów
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

# Generujemy 40 punktów z funkcji sinusoidalnej + szum losowy
n_samples = 40
X = np.sort(np.random.uniform(0, 4, n_samples))
# Prawdziwa funkcja świata (Ground Truth)
y_true = np.sin(X * 1.5) * 10 + 20
# Dane zebrane z rynku (z szumem pomiarowym)
y = y_true + np.random.normal(0, 2.5, n_samples)

# Podział na zbiór Treningowy (do nauki) i Testowy (do weryfikacji w świecie rzeczywistym)
X_train, X_test, y_train, y_test = train_test_split(X.reshape(-1, 1), y, test_size=0.3, random_state=42)

# Sortujemy zbiory do ładnego rysowania linii
sort_idx = X_test.flatten().argsort()
X_test_sorted = X_test[sort_idx]
y_test_sorted = y_test[sort_idx]

# Wizualizacja danych wyjściowych
plt.scatter(X_train, y_train, color='blue', s=60, label='Dane treningowe (Train)')
plt.scatter(X_test, y_test, color='red', s=60, marker='s', label='Dane testowe (Nowi klienci - Test)')
plt.plot(np.linspace(0, 4, 100), np.sin(np.linspace(0, 4, 100) * 1.5) * 10 + 20, 'k--', alpha=0.4, label='Prawdziwy trend (Niewidoczny w biznesie)')
plt.title("Rzeczywistość biznesowa: Pomiary z szumem oraz podział Train / Test", fontsize=14)
plt.xlabel("Cecha X (np. nakłady na reklamę w mln zł)")
plt.ylabel("Target Y (np. Przychód w mln zł)")
plt.legend()
plt.show()

In [ ]:
# ==============================================================================
# KROK 2: Porównanie wielomianów z dodanym SKALOWANIEM (StandardScaler)
# ==============================================================================
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

degrees = [1, 3, 14]
models = {}
X_plot = np.linspace(0, 4, 200).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for i, deg in enumerate(degrees):
    # Pipeline: Potęgi -> Standaryzacja -> Zwykła regresja
    model = make_pipeline(
        PolynomialFeatures(degree=deg, include_bias=False),
        StandardScaler(),
        LinearRegression()
    )
    model.fit(X_train, y_train)
    models[deg] = model
    
    # Błędy MSE
    mse_train = mean_squared_error(y_train, model.predict(X_train))
    mse_test = mean_squared_error(y_test, model.predict(X_test))
    
    # Wykres
    ax = axes[i]
    ax.scatter(X_train, y_train, color='blue', s=40, label='Train')
    ax.scatter(X_test, y_test, color='red', s=40, marker='s', label='Test')
    ax.plot(X_plot, model.predict(X_plot), color='green', lw=2.5, label=f'Model (stopień {deg})')
    
    nazwa_zjawiska = "1. UNDERFITTING (Bias)" if deg == 1 else ("2. DOBRE DOPASOWANIE" if deg == 3 else "3. OVERFITTING (Variance)")
    ax.set_title(f"{nazwa_zjawiska}\nTrain MSE: {mse_train:.1f} | Test MSE: {mse_test:.1f}", fontsize=12)
    ax.set_ylim(5, 35)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# KROK 3: Sprawdzamy współczynniki przeuczonego modelu
# ==============================================================================
overfitted_regressor = models[14].named_steps['linearregression']
weights = overfitted_regressor.coef_

print("--- WSPÓŁCZYNNIKI MODELU WIELOMIANOWEGO 14. STOPNIA ---")
for potega, waga in enumerate(weights, start=1):
    print(f"Cecha x^{potega:<2}: waga = {waga:>15.2f}")

In [ ]:
# ==============================================================================
# KROK 4: Ridge i Lasso na przeskalowanym wielomianie 14. stopnia
# ==============================================================================
from sklearn.linear_model import Ridge, Lasso

# Ridge (L2)
ridge_model = make_pipeline(
    PolynomialFeatures(degree=14, include_bias=False),
    StandardScaler(),
    Ridge(alpha=10.0)
)
ridge_model.fit(X_train, y_train)

# Lasso (L1)
lasso_model = make_pipeline(
    PolynomialFeatures(degree=14, include_bias=False),
    StandardScaler(),
    Lasso(alpha=0.3, max_iter=10000)
)
lasso_model.fit(X_train, y_train)

# Wykres
plt.figure(figsize=(12, 6))
plt.scatter(X_train, y_train, color='blue', alpha=0.6, s=50, label='Train')
plt.scatter(X_test, y_test, color='red', alpha=0.6, s=50, marker='s', label='Test')

plt.plot(X_plot, models[14].predict(X_plot), 'r:', label='Zwykła Regresja OLS (Przeuczona)', lw=1.5)
plt.plot(X_plot, ridge_model.predict(X_plot), 'g-', label='Ridge (L2, alpha=10.0) - Wygładzona', lw=2.5)
plt.plot(X_plot, lasso_model.predict(X_plot), 'm--', label='Lasso (L1, alpha=0.3) - Uproszczona', lw=2.5)

plt.ylim(10, 35)
plt.title("Ratunek dzięki Regularizacji: Skalowanie + Ridge/Lasso poskromiły wagi!", fontsize=14)
plt.xlabel("Cecha X")
plt.ylabel("Target Y")
plt.legend()
plt.show()

print(f"MSE Test (Zwykła OLS): {mean_squared_error(y_test, models[14].predict(X_test)):.1f}")
print(f"MSE Test (Ridge L2):   {mean_squared_error(y_test, ridge_model.predict(X_test)):.1f}  <-- Piękna generalizacja!")
print(f"MSE Test (Lasso L1):   {mean_squared_error(y_test, lasso_model.predict(X_test)):.1f}  <-- Piękna generalizacja!")

In [ ]:
# ==============================================================================
# KROK 5: Porównanie wag w tej samej skali: Zwykła OLS vs Ridge vs Lasso
# ==============================================================================
import pandas as pd

# Pobieramy wagi z poszczególnych modeli z wnętrza pipeline'ów
ols_coefs = models[14].named_steps['linearregression'].coef_
ridge_coefs = ridge_model.named_steps['ridge'].coef_
lasso_coefs = lasso_model.named_steps['lasso'].coef_

# Tworzymy czytelną tabelę porównawczą
df_coefs = pd.DataFrame({
    'Cecha': [f'x^{i}' for i in range(1, 15)],
    'Zwykła OLS': ols_coefs,
    'Ridge (L2)': ridge_coefs,
    'Lasso (L1)': lasso_coefs
})

# Formatujemy liczby do czytelnego widoku
pd.set_option('display.float_format', lambda x: f'{x:12.2f}')
print("--- PORÓWNANIE WSPÓŁCZYNNIKÓW DLA KAŻDEJ POTĘGI (WSPÓLNA SKALA) ---")
print(df_coefs.to_string(index=False))

# Zliczamy aktywne cechy
niezerowe_ols = np.sum(np.abs(ols_coefs) > 0.01)
niezerowe_ridge = np.sum(np.abs(ridge_coefs) > 0.01)
niezerowe_lasso = np.sum(np.abs(lasso_coefs) > 0.01)

print(f"\nLiczba aktywnych cech (istotnych wag) w modelu:")
print(f" • Zwykła regresja OLS: {niezerowe_ols} z 14 (wszystkie wagi szaleją w setkach tysięcy)")
print(f" • Ridge (L2):          {niezerowe_ridge} z 14 (wagi ściągnięte do bezpiecznych wartości)")
print(f" • Lasso (L1):          {niezerowe_lasso} z 14 (WYEROWAŁO {14 - niezerowe_lasso} ZBĘDNYCH POTĘG!)")